In [ ]:
from pathlib import Path
import sys

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np

sys.path.insert(0, str(Path("..").resolve()))

from geomexp.clustering.clustering_base import ClusterResult
from geomexp.visualization import ClusterVisualizer, PlotStyle

_plasma = mpl.colormaps["plasma"]
_n_clusters = 9
_plasma_colors = [mpl.colors.to_hex(_plasma(i / (_n_clusters - 1))) for i in range(_n_clusters)]

style = PlotStyle(figsize=(10, 5), use_latex=True, fontsize=11, color_palette=_plasma_colors)
viz = ClusterVisualizer(style)

PLOT_DIR = Path("../plots/supporting")
PLOT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
centers = np.array([
    [0.18, 0.78],
    [0.62, 0.91],
    [0.88, 0.72],
    [0.08, 0.42],
    [0.45, 0.55],
    [0.78, 0.38],
    [0.30, 0.18],
    [0.65, 0.12],
    [0.92, 0.08],
])
K = len(centers)

r = 0.75
raw_directions = np.array([
    [ 0.8, -0.6],
    [-0.3,  0.95],
    [-0.9, -0.4],
    [ 0.7,  0.7],
    [-0.6, -0.8],
    [ 0.4,  0.9],
    [ 0.9, -0.4],
    [-0.5,  0.85],
    [-0.8,  0.6],
])
norms = np.linalg.norm(raw_directions, axis=1, keepdims=True)
index_vectors = r * raw_directions / norms

res = 2_500
x_lo, x_hi = -0.05, 1.05
y_lo, y_hi = -0.05, 1.05
xx, yy = np.meshgrid(np.linspace(x_lo, x_hi, res), np.linspace(y_lo, y_hi, res))
grid = np.column_stack([xx.ravel(), yy.ravel()])


def kmeans_costs(grid_pts, ctrs):
    return np.column_stack([np.sum((grid_pts - c) ** 2, axis=1) for c in ctrs])


def gke_costs(grid_pts, ctrs, indices):
    costs = np.empty((grid_pts.shape[0], len(ctrs)))
    for k, (c, u) in enumerate(zip(ctrs, indices)):
        residuals = grid_pts - c
        dists = np.linalg.norm(residuals, axis=1)
        costs[:, k] = 0.5 * dists ** 2 + 0.5 * dists * np.sum(u * residuals, axis=1)
    return costs


def draw_boundaries(ax, xx, yy, costs, color="black", linewidth=0.8):
    """Draw decision boundaries via pairwise cost-difference contours."""
    n_clusters = costs.shape[1]
    assignments = np.argmin(costs, axis=1).reshape(xx.shape)
    for i in range(n_clusters):
        for j in range(i + 1, n_clusters):
            mask_i = assignments == i
            mask_j = assignments == j
            if not (mask_i.any() and mask_j.any()):
                continue
            diff = (costs[:, i] - costs[:, j]).reshape(xx.shape)
            masked_diff = np.where(mask_i | mask_j, diff, np.nan)
            ax.contour(xx, yy, masked_diff, levels=[0.0],
                       colors=color, linewidths=linewidth)


centroid_color = "black"
centroid_size = 35
centroid_linewidth = 1.5
boundary_color = style.color_palette[0]
boundary_linewidth = 2

if style.use_latex:
    mpl.rcParams.update({"text.usetex": True, "font.family": "serif",
                         "font.size": style.fontsize})

fig, axes = plt.subplots(1, 2, figsize=style.figsize)

costs_km = kmeans_costs(grid, centers)
draw_boundaries(axes[0], xx, yy, costs_km, color=boundary_color, linewidth=boundary_linewidth)
axes[0].scatter(centers[:, 0], centers[:, 1], s=centroid_size, facecolors="none",
                edgecolors=centroid_color, linewidths=centroid_linewidth, zorder=5)
costs_gk = gke_costs(grid, centers, index_vectors)
draw_boundaries(axes[1], xx, yy, costs_gk, color=boundary_color, linewidth=boundary_linewidth)
axes[1].scatter(centers[:, 0], centers[:, 1], s=centroid_size, facecolors="none",
                edgecolors=centroid_color, linewidths=centroid_linewidth, zorder=5)

for ax in axes:
    ax.set_xlim(x_lo, x_hi)
    ax.set_ylim(y_lo, y_hi)
    ax.set_aspect("equal")
    ax.axis("off")

fig.tight_layout()
fig.savefig(PLOT_DIR / "voronoi_comparison.pdf", bbox_inches="tight")
plt.show()

In [ ]:
style.center_color = "black"
style.center_size = 15
style.center_linewidth = 0.8
style.arrow_color = "black"
style.arrow_linewidth = 0.8
style.arrow_head_width = 0.15
style.arrow_head_length = 0.2
style.contour_linewidth = 0.8
style.curve_linewidth = 0.8
boundary_color_contours = style.color_palette[0]
style.figsize = (4, 7)
style.dpi = 1000

center = np.array([0, 0])
direction = np.array([3, 1])
direction = direction / np.linalg.norm(direction)

c1 = np.array([0, 0.7])
c2 = np.array([-1, -0.2])
c3 = np.array([1, -0.2])

x_lim = (-2, 2)
y_lim = (-2, 2)

radii_demo = [0, 0.3, 0.6, 0.9]
levels = np.linspace(0.5, 4, 12)

fig, axes = plt.subplots(
    4, 2,
    figsize=style.figsize,
    constrained_layout=True,
    sharex=True,
    sharey=True,
    dpi=viz.style.dpi,
)

for row_idx, r in enumerate(radii_demo):
    ax_contour = axes[row_idx, 0]
    u = r * direction
    viz.plot_expectile_curves(
        center, u,
        ax=ax_contour, title="", cost_levels=levels,
    )

    ax_boundary = axes[row_idx, 1]
    u1 = r * np.array([0, 1])
    u2 = np.array([0, 0])
    u3 = np.array([0, 0])

    result = ClusterResult(
        assignments=np.array([]),
        centers=np.array([c1, c2, c3]),
        objective=0,
        n_iterations=0,
        converged=True,
        metadata={"indices": np.array([u1, u2, u3])},
    )

    X_dummy = np.array([[x_lim[0], y_lim[0]], [x_lim[1], y_lim[1]]], dtype=float)
    viz.plot_decision_boundaries(
        X_dummy, result, ax=ax_boundary, title="",
        boundary_color=boundary_color_contours,
        show_regions=False, show_points=False, show_centers=True,
        resolution=1000,
    )

    for ax in (ax_contour, ax_boundary):
        ax.grid(False)

    if r > 0:
        ax_boundary.annotate(
            "",
            xy=(c1[0] + u1[0], c1[1] + u1[1]),
            xytext=(c1[0], c1[1]),
            arrowprops={
                "arrowstyle": f"->,head_width={style.arrow_head_width},head_length={style.arrow_head_length}",
                "lw": style.arrow_linewidth,
                "color": style.arrow_color,
                "shrinkA": 0,
                "shrinkB": 0,
            },
            zorder=8,
        )

    ax_contour.set_xlim(x_lim)
    ax_contour.set_ylim(y_lim)
    ax_boundary.set_xlim(x_lim)
    ax_boundary.set_ylim(y_lim)

    xticks = range(x_lim[0], x_lim[1] + 1)
    yticks = range(y_lim[0], y_lim[1] + 1)
    ax_contour.set_xticks(xticks)
    ax_boundary.set_xticks(xticks)
    ax_contour.set_yticks(yticks)
    ax_boundary.set_yticks(yticks)

    if row_idx < len(radii_demo) - 1:
        ax_contour.tick_params(
            axis="x", which="both", bottom=False, labelbottom=False,
        )
        ax_boundary.tick_params(
            axis="x", which="both", bottom=False, labelbottom=False,
        )

    ax_boundary.tick_params(
        axis="y", which="both",
        left=False, labelleft=False, right=False, labelright=False,
    )

    ax_contour.set_xlabel("")
    ax_boundary.set_xlabel("")
    ax_contour.set_ylabel("")
    ax_boundary.set_ylabel("")

    for ax in (ax_contour, ax_boundary):
        ax.grid(False)
        ax.tick_params(labelsize=style.fontsize - 1, length=2, width=style.axis_linewidth)

fig.supxlabel(r"$x_1$", fontsize=style.fontsize)
fig.supylabel(r"$x_2$", fontsize=style.fontsize)
plt.show()

viz.save_figure(fig, str(PLOT_DIR / "expectile_contours.pdf"))